# Experiment: Landmark-Driven Place Cell Remapping

This notebook tests whether the Visual Place Cell (VPCE) ensemble trained on **LM8** exhibits **remapping** when the landmarks are rotated.

**LM8** and **LM8_R45** share the identical octagon boundary and obstacle layout. The only difference is that each landmark panel in LM8_R45 is shifted forward by one wall (45°), so the robot sees a different flag texture at every wall position.

---

## Hypothesis

If the place cells encode the visual scene (landmark identity + geometry) rather than raw position, their activation fields should **rotate with the landmarks** — cells that previously fired in one region of the maze should shift to the adjacent region when the landmarks rotate by 45°.

---

## Pipeline

```
data/vpce/place_cells/lm8.h5          ← place cell model trained on LM8
        +
data/vpce/collect_data/lm8_r45.h5     ← LM8_R45 observations (test set)
        ↓
  Compute activations using LM8 place cells on LM8_R45 features
        ↓
  Activation projection onto LM8_R45 x,y plane
```

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import xml.etree.ElementTree as ET
os.chdir('..')  # set working directory to project root

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

from realm_tools.experiment_lib.loggers import PovDataset, PlaceCellEnsemble
from realm_tools.place_cell_lib import VisualPlaceCellEnsemble
from realm_tools.image_lib.analysis_plots import (
    plot_place_cell_activations,
    plot_place_cell_activations_overlay,
    _draw_maze,
)
from realm_tools.simulation_lib.environment_parser import parse_all_walls

---
## Configuration

In [ ]:
TRAIN_MAZE      = 'lm8'       # maze the place cells were trained on
TEST_MAZE       = 'lm8_r45'   # maze with rotated landmarks (test condition)

PLACE_CELL_PATH  = f'data/vpce/place_cells/{TRAIN_MAZE}'
TRAIN_DATA_PATH  = f'data/vpce/collect_data/{TRAIN_MAZE}'
TEST_DATA_PATH   = f'data/vpce/collect_data/{TEST_MAZE}'
TRAIN_MAZE_XML   = f'simulation/worlds/environments/vpce/{TRAIN_MAZE}.xml'
TEST_MAZE_XML    = f'simulation/worlds/environments/vpce/{TEST_MAZE}.xml'
TEST_BASE_IMG    = f'analysis/base_figures/{TEST_MAZE}.png'

# Cells to include in the side-by-side comparison.
# Pass a list of specific cell indices, or set to None to show the first N_COMPARE cells.
COMPARE_CELLS = [0, 19, 23, 29, 30]
N_COMPARE     = 20   # used only when COMPARE_CELLS is None

---
## Load Place Cell Model

The ensemble (centers + radii) was produced by `clustering.ipynb` and saved to `data/vpce/place_cells/lm8.h5`.

In [ ]:
pc_model = PlaceCellEnsemble.load(PLACE_CELL_PATH)
ensemble = VisualPlaceCellEnsemble(pc_model.centers, pc_model.radii)

print(f"Loaded  : {PLACE_CELL_PATH}")
print(f"Method  : {pc_model.method}")
print(f"Maze    : {pc_model.maze}")
print(f"Cells   : {ensemble.n_cells}")
print(f"Feat dim: {ensemble.feature_dim}")

In [ ]:
# Load LM8_R45 observations
test_dataset  = PovDataset.load_dataset(TEST_DATA_PATH)
features_test = np.array(test_dataset.multimodal_features)
poses_test    = np.stack([test_dataset.x, test_dataset.y, test_dataset.theta], axis=1)

print(f"Observations : {features_test.shape[0]}  ({TEST_MAZE.upper()})")
print(f"Feature dim  : {features_test.shape[1]}")

---
## Compute Activations

Apply the LM8 place cell ensemble to the LM8_R45 observations.  Each row of the resulting matrix is the normalised population activation vector for one observation position in the rotated-landmark environment.

In [ ]:
activations = ensemble.activate(features_test)
print(f"Activation matrix : {activations.shape}  (observations × cells)")

---
## Activation Projection — LM8_R45

Each subplot shows the activation field of one LM8 place cell measured in the **rotated-landmark environment**.  Compare with the equivalent plot from `clustering.ipynb` (LM8 activations measured in LM8) — if remapping has occurred, the activation peaks should be shifted relative to their original positions.

In [ ]:
plot_place_cell_activations(
    activations      = activations,
    poses            = poses_test,
    maze_xml         = TEST_MAZE_XML,
    maze             = TEST_MAZE,
    method           = pc_model.method,
    save             = True

)

In [ ]:
plot_place_cell_activations_overlay(
    activations   = activations,
    poses         = poses_test,
    maze_xml      = TEST_MAZE_XML,
    base_img_path = TEST_BASE_IMG,
    maze          = TEST_MAZE,
    method        = pc_model.method,
    flip_base_img = True,
    save          = False,
)

---
## Cell-by-Cell Comparison: LM8 vs LM8_R45

Each row shows one place cell. The **left** column is the cell's activation field measured in the original LM8 environment (baseline). The **right** column is the same cell measured in LM8_R45 (landmarks rotated 45°).

Both columns share the same colour scale per cell so the relative strength of activation is directly comparable. If the place cells encode visual landmarks, the activation peak in the right column should be shifted ≈ 45° relative to the left.

In [ ]:
# --- Load LM8 baseline data and compute activations ---
train_dataset   = PovDataset.load_dataset(TRAIN_DATA_PATH)
features_train  = np.array(train_dataset.multimodal_features)
poses_train     = np.stack([train_dataset.x, train_dataset.y, train_dataset.theta], axis=1)

activations_train = ensemble.activate(features_train)

# --- Resolve which cells to show ---
cell_indices = COMPARE_CELLS if COMPARE_CELLS is not None else list(range(min(N_COMPARE, ensemble.n_cells)))
n_show = len(cell_indices)

# --- Build interpolation grids ---
margin, resolution = 0.1, 200

def make_grid(poses):
    x, y = poses[:, 0], poses[:, 1]
    xi = np.linspace(x.min() - margin, x.max() + margin, resolution)
    yi = np.linspace(y.min() - margin, y.max() + margin, resolution)
    return np.meshgrid(xi, yi), xi, yi

(Xi_tr, Yi_tr), xi_tr, yi_tr = make_grid(poses_train)
(Xi_te, Yi_te), xi_te, yi_te = make_grid(poses_test)

# --- Layout ---
cell_h = 4.0
cmap   = plt.get_cmap('inferno').copy()
cmap.set_bad('white')

fig, axes = plt.subplots(n_show, 2,
                          figsize=(cell_h * 2 + 1.5, cell_h * n_show + 1.5),
                          facecolor='white')
if n_show == 1:
    axes = axes[np.newaxis, :]

fig.suptitle(
    f'Place Cell Remapping: {TRAIN_MAZE.upper()} → {TEST_MAZE.upper()}\n'
    f'LM8 place cells ({pc_model.method.upper()}, K={ensemble.n_cells}) '
    f'activated by each environment',
    fontsize=16, color='black', y=1.0, fontweight='bold'
)

# Column headers — descriptive
axes[0, 0].set_title(f'{TRAIN_MAZE.upper()}\n(trained environment — baseline)',
                     fontsize=13, color='black', pad=8)
axes[0, 1].set_title(f'{TEST_MAZE.upper()}\n(landmarks rotated +45°)',
                     fontsize=13, color='black', pad=8)

for row, cell_idx in enumerate(cell_indices):
    ax_l, ax_r = axes[row, 0], axes[row, 1]

    Zi_tr = griddata((poses_train[:, 0], poses_train[:, 1]),
                     activations_train[:, cell_idx], (Xi_tr, Yi_tr), method='linear')
    Zi_te = griddata((poses_test[:, 0],  poses_test[:, 1]),
                     activations[:, cell_idx],        (Xi_te, Yi_te), method='linear')

    for ax, Zi, xi, yi, xml, act_col in [
        (ax_l, Zi_tr, xi_tr, yi_tr, TRAIN_MAZE_XML, activations_train[:, cell_idx]),
        (ax_r, Zi_te, xi_te, yi_te, TEST_MAZE_XML,  activations[:, cell_idx]),
    ]:
        ax.set_facecolor('white')
        im = ax.imshow(Zi, extent=[xi.min(), xi.max(), yi.min(), yi.max()],
                       origin='lower', cmap=cmap, vmin=0, vmax=act_col.max(),
                       aspect='equal', interpolation='bilinear')
        _draw_maze(ax, xml)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect('equal')

    ax_l.set_ylabel(f'Cell {cell_idx}', fontsize=12, color='black',
                    rotation=0, labelpad=45, va='center', fontweight='bold')

plt.subplots_adjust(hspace=0.08, wspace=0.05, top=0.95, bottom=0.01,
                    left=0.12, right=0.98)
plt.show()